# Modelado y Línea Base (Bank Marketing)

**Grupo:** Grupo 4  
**Archivo del módulo:** `src/modelo.py`  
**Objetivo del notebook:** Implementar los modelos base (trivial y simple) y las Redes Neuronales (MLP v1 y v2), generando la tabla comparativa oficial de la Etapa 3.

In [38]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto al path
raiz_proyecto = Path("..").resolve()
if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

# Importar desde el archivo src/modelo.py limpio (sin tildes)
from src.modelo import (
    entrenar_modelo_trivial,
    entrenar_baseline_simple,
    construir_mlp_v1,
    construir_mlp_v2
)

print("¡Librerías y módulos importados correctamente sin errores!")

¡Librerías y módulos importados correctamente sin errores!


### Carga y Partición de Datos
Cargamos el dataset procesado y aplicamos el pipeline de partición para mantener el aislamiento estricto entre entrenamiento y validación.

In [39]:
from src.datos import cargar_y_limpiar_datos, preparar_particiones

ruta_csv = Path("../data/bank-full.csv")
X, y = cargar_y_limpiar_datos(ruta_csv)
X_train, X_val, X_test, y_train, y_val, y_test, scaler = preparar_particiones(X, y)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_val:   {X_val.shape}")

Dimensiones de X_train: (31650, 41)
Dimensiones de X_val:   (6779, 41)


### Entrenamiento de Modelos Base y Redes Neuronales (Etapa 3)
Entrenaremos cuatro enfoques diferentes:
1. **Modelo Trivial (Dummy):** Predice siempre la clase mayoritaria.
2. **Baseline Simple (Regresión Logística):** Modelo estadístico con clases balanceadas.
3. **Red Neuronal (MLP V1):** Perceptrón multicapa básico con una capa oculta.
4. **Red Neuronal (MLP V2 + Dropout):** Perceptrón más profundo con prevención de sobreajuste.

In [40]:
resultados = []
input_dim = X_train.shape[1]
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# --- 1. MODELO TRIVIAL ---
print("Entrenando Modelo Trivial...")
modelo_trivial = entrenar_modelo_trivial(X_train, y_train)
y_pred_trivial = modelo_trivial.predict(X_val)
y_prob_trivial = modelo_trivial.predict_proba(X_val)[:, 1]

resultados.append({
    "Modelo": "1. Trivial (Dummy)",
    "Accuracy": accuracy_score(y_val, y_pred_trivial),
    "Precision": precision_score(y_val, y_pred_trivial, zero_division=0),
    "Recall": recall_score(y_val, y_pred_trivial, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_trivial, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_trivial)
})

# --- 2. MODELO SIMPLE (Regresión Logística) ---
print("Entrenando Baseline Simple...")
modelo_simple = entrenar_baseline_simple(X_train, y_train)
y_pred_simple = modelo_simple.predict(X_val)
y_prob_simple = modelo_simple.predict_proba(X_val)[:, 1]

resultados.append({
    "Modelo": "2. Baseline Simple (LogReg)",
    "Accuracy": accuracy_score(y_val, y_pred_simple),
    "Precision": precision_score(y_val, y_pred_simple, zero_division=0),
    "Recall": recall_score(y_val, y_pred_simple, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_simple, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_simple)
})

# --- 3. MLP V1 ---
print("Entrenando Red Neuronal - MLP Versión 1...")
mlp_v1 = construir_mlp_v1(input_dim)
mlp_v1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

y_prob_v1 = mlp_v1.predict(X_val).ravel()
y_pred_v1 = (y_prob_v1 >= 0.5).astype(int)

resultados.append({
    "Modelo": "3. Red Neuronal (MLP V1)",
    "Accuracy": accuracy_score(y_val, y_pred_v1),
    "Precision": precision_score(y_val, y_pred_v1, zero_division=0),
    "Recall": recall_score(y_val, y_pred_v1, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_v1, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_v1)
})

# --- 4. MLP V2 (Con Dropout) ---
print("Entrenando Red Neuronal - MLP Versión 2 (Dropout)...")
mlp_v2 = construir_mlp_v2(input_dim)
mlp_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

y_prob_v2 = mlp_v2.predict(X_val).ravel()
y_pred_v2 = (y_prob_v2 >= 0.5).astype(int)

resultados.append({
    "Modelo": "4. Red Neuronal (MLP V2 + Dropout)",
    "Accuracy": accuracy_score(y_val, y_pred_v2),
    "Precision": precision_score(y_val, y_pred_v2, zero_division=0),
    "Recall": recall_score(y_val, y_pred_v2, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_v2, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_v2)
})

print("¡Todos los modelos han sido entrenados con éxito!")

Entrenando Modelo Trivial...
Entrenando Baseline Simple...
Entrenando Red Neuronal - MLP Versión 1...
212/212 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Entrenando Red Neuronal - MLP Versión 2 (Dropout)...
212/212 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
¡Todos los modelos han sido entrenados con éxito!


### Tabla Comparativa de Rendimiento
Generamos la tabla oficial con las métricas clave de evaluación en el conjunto de validación (`Accuracy`, `Precision`, `Recall`, `F1-Score` y `ROC-AUC`).

In [41]:
df_comparacion = pd.DataFrame(resultados).set_index("Modelo")

print("="*65)
print(" TABLA COMPARATIVA DE RENDIMIENTO DE MODELOS (ETAPA 3) ")
print("="*65)
display(df_comparacion.round(4))

 TABLA COMPARATIVA DE RENDIMIENTO DE MODELOS (ETAPA 3) 


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Modelo,,,,,
1. Trivial (Dummy),0.8830,0.0000,0.0000,0.0000,0.5000
2. Baseline Simple (LogReg),0.7551,0.2690,0.6368,0.3783,0.7720
3. Red Neuronal (MLP V1),0.8970,0.6678,0.2383,0.3513,0.7978
4. Red Neuronal (MLP V2 + Dropout),0.8933,0.6316,0.2119,0.3173,0.8004
